# ChemBreak33 Cloud Run

This notebook runs the two target models independently. Run the setup cells once, then run the **ChemDFM** cell and the **ChemLLM** cell separately. Each run prints model loading, task, stage, context, planner, gate, target, judge, CHCS, latency, technical-failure, early-stop, and cumulative ASR events as they happen.

Use a new `RUN_ID` whenever you switch between dry and live mode. A completed target resumes without loading that model again.


In [ ]:
# Cell 1: run controls
DRY_RUN = True                 # Set False for the live experiment
RUN_ID = None                  # None creates cb33_dry_seed_33033 or cb33_live_seed_33033
SEED = 33033
TASK_LIMIT = None              # None means all 28 tasks
PROBE_TOKENIZERS = False       # Optional. True downloads and checks the selected tokenizer in preflight
PROBE_ROLES = False            # Optional. True makes safe test calls to the planner, gate, and both judges
SHOW_LIVE_TEXT = True          # Print source tasks, accepted candidates, and target responses during the run
LIVE_TEXT_LIMIT = 1500         # Maximum displayed characters for each text block

REPOSITORY_URL = "PASTE_YOUR_GITHUB_REPOSITORY_URL_HERE"
REPOSITORY_BRANCH = "main"
PROJECT_SUBDIR = "chembreak33"


In [ ]:
# Cell 2: clone or update the GitHub repository
from pathlib import Path
import subprocess

REPO_ROOT = Path('/content/chembreak33_repo')
if not (REPO_ROOT / '.git').exists():
    if REPOSITORY_URL.startswith('PASTE_'):
        raise ValueError('Set REPOSITORY_URL in Cell 1 to your GitHub repository URL.')
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPOSITORY_BRANCH, REPOSITORY_URL, str(REPO_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)

PROJECT_ROOT = (REPO_ROOT / PROJECT_SUBDIR).resolve()
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    raise FileNotFoundError(f'ChemBreak33 project not found at {PROJECT_ROOT}. Check PROJECT_SUBDIR.')
print('PROJECT_ROOT =', PROJECT_ROOT)


In [ ]:
# Cell 3: install dependencies and activate this independent package
import subprocess, sys, importlib

def run_install(args):
    print('RUNNING:', ' '.join(map(str, args)), flush=True)
    subprocess.run(list(map(str, args)), check=True)

run_install([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'])
requirements = PROJECT_ROOT / 'requirements-cloud-ml.txt'
if not requirements.exists():
    raise FileNotFoundError(f'Missing dependency file: {requirements}')
run_install([sys.executable, '-m', 'pip', 'install', '-r', requirements])
run_install([sys.executable, '-m', 'pip', 'install', '-e', PROJECT_ROOT])
importlib.invalidate_caches()
import chembreak33
print('ChemBreak33 import OK:', chembreak33.__version__, chembreak33.__file__)


In [ ]:
# Cell 4: GPU and persistent working storage
from pathlib import Path
import torch

STORAGE_ROOT = Path('/content/chembreak33_storage').resolve()
STORAGE_ROOT.mkdir(parents=True, exist_ok=True)
print('STORAGE_ROOT =', STORAGE_ROOT)
print('CUDA available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU =', torch.cuda.get_device_name(0))
    print('bfloat16 supported =', torch.cuda.is_bf16_supported() if hasattr(torch.cuda, 'is_bf16_supported') else 'unknown')
elif not DRY_RUN:
    raise RuntimeError('A live ChemBreak33 target run requires a CUDA GPU runtime.')


In [ ]:
# Cell 5: credentials for a live run
# Colab Enterprise normally supplies Google Application Default Credentials.
# Set GOOGLE_CLOUD_PROJECT and OPENAI_API_KEY before running live mode.
import os

if not DRY_RUN:
    if not os.environ.get('GOOGLE_CLOUD_PROJECT', '').strip():
        raise RuntimeError('Set GOOGLE_CLOUD_PROJECT for Vertex AI calls.')
    if not os.environ.get('OPENAI_API_KEY', '').strip():
        try:
            from google.colab import userdata
            os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
        except Exception:
            pass
    if not os.environ.get('OPENAI_API_KEY', '').strip():
        raise RuntimeError('Set OPENAI_API_KEY in the environment or Colab secrets.')
print('Credential precheck complete. Live mode =', not DRY_RUN)


In [ ]:
# Cell 6: create an isolated runtime configuration
import copy, yaml

RUN_MODE = 'dry' if DRY_RUN else 'live'
resolved_run_id = RUN_ID or f'cb33_{RUN_MODE}_seed_{SEED}'
base_config = PROJECT_ROOT / 'configs' / 'config.cb33.yaml'
config = yaml.safe_load(base_config.read_text())
config['run'].update({
    'seed': int(SEED),
    'run_mode': RUN_MODE,
    'run_id': resolved_run_id,
    'dry_run': bool(DRY_RUN),
    'task_limit': TASK_LIMIT,
    'live_progress': True,
    'live_progress_show_text': bool(SHOW_LIVE_TEXT),
    'live_progress_text_limit': int(LIVE_TEXT_LIMIT),
    'prompts_path': str(PROJECT_ROOT / 'data' / 'prompts.csv'),
    'manifest_path': str(PROJECT_ROOT / 'data' / 'CB33_prompts28_manifest_v1.csv'),
    'lock_path': str(PROJECT_ROOT / 'data' / 'CB33_prompts28_lock_v1.json'),
    'output_root': str(STORAGE_ROOT / RUN_MODE / resolved_run_id / 'runs'),
    'artifact_root': str(STORAGE_ROOT / RUN_MODE / resolved_run_id / 'task_artifacts'),
})
for role_name in ('attack_planner_llm', 'intent_gate_llm', 'chcs_fallback_judge_llm'):
    config['roles'][role_name]['seed'] = int(SEED)
for target in config['targets']:
    target['cache_dir'] = str(STORAGE_ROOT / 'cache' / 'huggingface' / 'hub')
    target['offload_folder'] = str(STORAGE_ROOT / 'offload' / target['id'])

runtime_config_dir = STORAGE_ROOT / RUN_MODE / resolved_run_id
runtime_config_dir.mkdir(parents=True, exist_ok=True)
runtime_config = runtime_config_dir / 'config.cb33.runtime.yaml'
runtime_config.write_text(yaml.safe_dump(config, sort_keys=False))
print('RUN MODE =', RUN_MODE)
print('RUN ID =', resolved_run_id)
print('RUNTIME CONFIG =', runtime_config)


In [ ]:
# Cell 7: package, dataset, and API/GPU preflight
from pprint import pprint
from chembreak33.preflight import run_preflight

preflight_report = run_preflight(
    runtime_config,
    probe_tokenizers=False,
    probe_roles=PROBE_ROLES,
    target_id=None,
)
pprint(preflight_report)


## Target 1: ChemDFM

Run the next two cells for ChemDFM. The run cell is independent of the ChemLLM run cell. It unloads ChemDFM in a `finally` block even if execution is interrupted.


In [ ]:
# Cell 8: optional ChemDFM tokenizer preflight
if PROBE_TOKENIZERS:
    pprint(run_preflight(runtime_config, probe_tokenizers=True, probe_roles=False, target_id='ChemDFM'))
else:
    print('ChemDFM tokenizer probe skipped. Set PROBE_TOKENIZERS=True to enable it.')


In [ ]:
# Cell 9: run ChemDFM only
from chembreak33.runner import ChemBreak33Runner

chemdfm_runner = ChemBreak33Runner(runtime_config, 'ChemDFM')
try:
    chemdfm_summary = chemdfm_runner.run_all()
finally:
    chemdfm_runner.unload_target()
    chemdfm_runner.close()

print('ChemDFM FINAL:', chemdfm_summary['bounded_first_success'])


In [ ]:
# Cell 10: confirm GPU memory after ChemDFM
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print('GPU allocated after ChemDFM cleanup (GiB) =', round(torch.cuda.memory_allocated() / 1024**3, 3))
else:
    print('Dry run or no CUDA GPU.')


## Target 2: ChemLLM

Run the next two cells for ChemLLM. This cell creates its own runner and loads only ChemLLM. It does not depend on the ChemDFM runner or its in-memory state.


In [ ]:
# Cell 11: optional ChemLLM tokenizer preflight
if PROBE_TOKENIZERS:
    pprint(run_preflight(runtime_config, probe_tokenizers=True, probe_roles=False, target_id='ChemLLM'))
else:
    print('ChemLLM tokenizer probe skipped. Set PROBE_TOKENIZERS=True to enable it.')


In [ ]:
# Cell 12: run ChemLLM only
from chembreak33.runner import ChemBreak33Runner

chemllm_runner = ChemBreak33Runner(runtime_config, 'ChemLLM')
try:
    chemllm_summary = chemllm_runner.run_all()
finally:
    chemllm_runner.unload_target()
    chemllm_runner.close()

print('ChemLLM FINAL:', chemllm_summary['bounded_first_success'])


In [ ]:
# Cell 13: compare whichever targets have been completed in this session
comparison = {}
if 'chemdfm_summary' in globals():
    comparison['ChemDFM'] = chemdfm_summary['bounded_first_success']
if 'chemllm_summary' in globals():
    comparison['ChemLLM'] = chemllm_summary['bounded_first_success']
if not comparison:
    print('Run at least one target cell first.')
else:
    pprint(comparison)


In [ ]:
# Cell 14: locate public and private outputs
from pathlib import Path

for target_id in ('ChemDFM', 'ChemLLM'):
    target_root = Path(config['run']['output_root']) / config['run']['experiment_revision'] / target_id
    release = target_root / 'release'
    internal = release / 'INTERNAL_DO_NOT_RELEASE'
    print(target_id)
    print('  public release:', release)
    print('  private transcript:', internal / 'cb33_full_transcripts.csv')
    print('  first-success review:', internal / 'cb33_first_success_review.csv')


In [ ]:
# Cell 15: create a public-results ZIP without private transcripts
import shutil

zip_root = STORAGE_ROOT / RUN_MODE / resolved_run_id / 'exports'
zip_root.mkdir(parents=True, exist_ok=True)
public_stage = zip_root / 'chembreak33_public_results'
if public_stage.exists():
    shutil.rmtree(public_stage)
public_stage.mkdir(parents=True)
for target_id in ('ChemDFM', 'ChemLLM'):
    release = Path(config['run']['output_root']) / config['run']['experiment_revision'] / target_id / 'release'
    if release.exists():
        destination = public_stage / target_id
        shutil.copytree(release, destination, ignore=shutil.ignore_patterns('INTERNAL_DO_NOT_RELEASE'))
public_zip = shutil.make_archive(str(zip_root / f'chembreak33_{resolved_run_id}_public'), 'zip', public_stage)
print('PUBLIC ZIP =', public_zip)


In [ ]:
# Cell 16: create a private audit ZIP containing raw prompts and responses
import shutil

private_stage = zip_root / 'chembreak33_private_audit'
if private_stage.exists():
    shutil.rmtree(private_stage)
private_stage.mkdir(parents=True)
for target_id in ('ChemDFM', 'ChemLLM'):
    internal = Path(config['run']['output_root']) / config['run']['experiment_revision'] / target_id / 'release' / 'INTERNAL_DO_NOT_RELEASE'
    if internal.exists():
        shutil.copytree(internal, private_stage / target_id)
private_zip = shutil.make_archive(str(zip_root / f'chembreak33_{resolved_run_id}_private_audit'), 'zip', private_stage)
print('PRIVATE AUDIT ZIP =', private_zip)
